# Uncertainty GEPA: pipeline e tabelas do paper

| modulo | conteudo |
|---|---|
| `config.py` | backend (cloudflare / openrouter / ollama), modelos, constantes, diretorios |
| `benchmarks.py` | registry de benchmarks e loaders |
| `llm.py` | chamadas HTTP, LLM 1 (anotador), LLM 2 (assessor), LLM 3 (mestre) |
| `metrics.py` | Spearman, bootstrap, teto da escala, folds |
| `prompts.py` | prompt semente, baselines, template de reflexao |
| `gepa_search.py` | adapter e busca do GEPA |
| `pipeline.py` | carrega corpora, estima custo, roda o LLM 1 |
| `evaluation.py` | avaliacao no teste e teste de ruido |
| `paper.py` | escreve os `.tex` e o zip |

Saidas em `../paper_materials/{work,results,paper}`. Com tudo em cache (LLM 1 e codebooks), as
secoes 1 a 3 nao chamam LLM e dá para pular direto para a 4 e 6.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
sys.path.insert(0, str(_here if _here.name == "scripts" else _here / "scripts"))

# Colab: !pip install -q gepa datasets
import config as C, benchmarks, pipeline, gepa_search, evaluation, paper

C.configure()     # padrao: $BACKEND ou "ollama". Ou: C.configure("openrouter") / C.configure("cloudflare")
# Experimento: saidas isoladas em paper_materials/<EXPERIMENT>/{work,results,paper}, para nao
# reaproveitar codebooks nem sobrescrever o summary de outro experimento.
EXPERIMENT = "gpt56luna"
C.set_experiment(EXPERIMENT)

# LLM 1: resultados ja gerados em paper_materials/results/llm1_<tag>_<benchmark>.csv (R=20 reamostragens
# + rotulo). Com LLM1_TAG = None o pipeline roda o anotador configurado em config.py.
LLM1_TAG = "gpt56luna"
RUN = ["ArMIS", "HSBrexit", "ConvAbuse"]     # benchmarks com llm1 pronto; ou benchmarks.RUN
print("saidas em:", C.OUTDIR / EXPERIMENT)
print("backend:", C.BACKEND, "| assessor:", C.URLS["assessor"], "| workers:", C.WORKERS)

## 1. Dados e custo

So rede para baixar os corpora, nenhuma chamada a LLM.

In [ ]:
data = pipeline.load_corpora(RUN)
cost = pipeline.estimate_cost(data, llm1_given=bool(LLM1_TAG))

## 2. LLM 1: rotulo e alvo de instabilidade `g_m`

Com `LLM1_TAG` definido, le os resultados ja gerados (rotulo + 20 reamostragens por item; `g_m` e a
variancia das 20 extracoes) e **nao chama nenhuma LLM**. Texto, `u_m` e `consensus` vem dos loaders.
Com `LLM1_TAG = None`, roda o anotador de `config.py`, com cache em disco.

In [ ]:
frames = pipeline.build_frames(data, llm1_tag=LLM1_TAG)

## 3. GEPA: um codebook por benchmark e condicao

Reaproveita `work/codebook_*.txt`. Use um backend capaz para o mestre (openrouter ou cloudflare).

In [ ]:
print(gepa_search.smoke_test_master())

In [ ]:
codebooks = gepa_search.optimize_codebooks(frames, RUN)

## 4. Avaliacao no teste, todos os metodos

In [ ]:
summary, per_item = evaluation.evaluate_test(frames, codebooks, RUN)
summary

## 5. (opcional) Ruido do assessor a temperatura 0

Repete o mesmo prompt `k` vezes (condicao blind) num benchmark e mede quanto o hedge muda.
`provider="keep"` usa o provedor de `C.OR_PROVIDER["assessor"]`; `None` deixa o roteador escolher.

In [ ]:
noise_runs, noise_summary = evaluation.noise_test("HSBrexitOff", k=3, provider="keep")
noise_summary

## 6. Tabelas do paper

Escreve `paper_materials/paper/*.tex`. `summary` e `codebooks` caem para o que estiver em disco se voce nao os passar.

In [ ]:
paper.write_all(frames, summary, codebooks, RUN)
# so a partir do disco, sem rerodar as secoes 4 e 5:  paper.write_all(frames)

In [ ]:
# rho(h, u) por metodo, condicao blind
summary[summary.condition == "blind"].pivot(index="method", columns="benchmark", values="rho_u").round(3)

In [ ]:
print(paper.show("tab_main.tex"))

In [ ]:
paper.zip_materials()